# CommitBench filtering

Loads the CommitBench dataset, filters it down to good training examples, and writes
`train.jsonl` / `validation.jsonl` / `test.jsonl`.

Filtering policy:
- drop diffs touching lockfiles / generated / minified files
- keep only messages matching Conventional Commits (`feat:`, `fix:`, `refactor:`, ...)
- drop low-content messages ("fix", "wip", "update", too short, ...)
- truncate oversized diffs per-file, prioritizing hunks over full context

**Expect a low keep rate.** Raw CommitBench messages are real-world GitHub commits,
not Conventional-Commits-formatted — only ~5% naturally match a `feat:`/`fix:`/...
prefix. That's expected, not a bug: with ~1.17M train rows this still yields tens
of thousands of clean examples, plenty for LoRA fine-tuning.

**On Kaggle:** Settings -> Internet -> On (required to download the dataset and
packages). No GPU/accelerator needed for this notebook.

This notebook is self-contained (no dependency on the local `commit_model` package)
so it can run anywhere — locally, on Kaggle, or on Colab. Upload the resulting
`*.jsonl` files as input to `fine-tuning_Kaggle.ipynb`.

In [ ]:
# --- Dependencies ---
%pip install -q -U datasets tqdm

import json
import os
import re
import shutil
from pathlib import Path

from datasets import load_dataset
from tqdm.auto import tqdm

In [ ]:
# --- Config & filtering logic ---

# Kaggle only allows writes under /kaggle/working; detect it so this notebook
# also works unchanged when run locally or on Colab.
ON_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = Path("/kaggle/working/data/processed") if ON_KAGGLE else Path("data/processed")
LIMIT_PER_SPLIT = None  # set an int for quick iteration, e.g. 2000

CONVENTIONAL_PREFIX = re.compile(
    r"^(feat|fix|refactor|chore|docs|test|perf|style|build|ci)(\([\w./-]+\))?!?:\s+\S"
)

LOW_CONTENT_STOPLIST = {
    "fix", "wip", "update", "updates", "minor", "minor changes", "misc",
    "cleanup", "small fix", "typo", "changes", "stuff", "test", "tmp",
    "asdf", "oops", "fixup",
}

GENERATED_FILE_PATTERNS = [
    re.compile(p) for p in [
        r"package-lock\.json$", r"yarn\.lock$", r"pnpm-lock\.yaml$",
        r"Cargo\.lock$", r"Gemfile\.lock$", r"poetry\.lock$", r"uv\.lock$",
        r"\.min\.(js|css)$", r"\.map$", r"/dist/", r"/build/", r"/vendor/",
        r"\.svg$", r"\.snap$",
    ]
]

MAX_DIFF_CHARS = 4000
MAX_FILES_PER_DIFF = 6


def is_generated_path(path: str) -> bool:
    return any(p.search(path) for p in GENERATED_FILE_PATTERNS)


def message_is_low_content(message: str) -> bool:
    first_line = message.strip().splitlines()[0] if message.strip() else ""
    has_prefix = CONVENTIONAL_PREFIX.match(first_line)
    subject = first_line.split(":", 1)[-1].strip() if has_prefix else first_line
    normalized = subject.lower().strip(" .!")
    return len(normalized) < 8 or normalized in LOW_CONTENT_STOPLIST


def split_diff_by_file(diff: str):
    """Split a unified diff into (path, hunk_text) chunks."""
    chunks = re.split(r"(?=^diff --git )", diff, flags=re.MULTILINE)
    files = []
    for chunk in chunks:
        if not chunk.strip():
            continue
        match = re.search(r"^diff --git a/(\S+) b/(\S+)", chunk)
        path = match.group(2) if match else "unknown"
        files.append((path, chunk))
    return files


def truncate_diff(diff: str):
    """Drop generated/lockfile paths, cap file count, and cap total length."""
    files = split_diff_by_file(diff)
    kept = [(p, c) for p, c in files if not is_generated_path(p)]
    if not kept:
        return None
    kept = kept[:MAX_FILES_PER_DIFF]

    pieces = []
    for path, chunk in kept:
        share = max(MAX_DIFF_CHARS // max(len(kept), 1), 200)
        pieces.append(chunk[:share])
    return "".join(pieces)[:MAX_DIFF_CHARS]


def build_example(diff: str, message: str):
    message = message.strip()
    if not message:
        return None
    first_line = message.splitlines()[0]
    if not CONVENTIONAL_PREFIX.match(first_line):
        return None
    if message_is_low_content(message):
        return None
    truncated = truncate_diff(diff)
    if truncated is None or len(truncated.strip()) < 20:
        return None
    return {"diff": truncated, "message": first_line}


# --- Load ---
print(f"Running on Kaggle: {ON_KAGGLE}")
print(f"Output dir: {OUT_DIR.resolve()}")

try:
    ds = load_dataset("Maxscha/commitbench")
except Exception as e:
    raise RuntimeError(
        "Failed to download CommitBench. On Kaggle, make sure Settings -> Internet "
        "is turned ON, then re-run this cell."
    ) from e


# --- Filter & write ---
OUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name in ds.keys():
    split = ds[split_name]
    out_path = OUT_DIR / f"{split_name}.jsonl"
    kept, seen = 0, 0
    with out_path.open("w") as f:
        for row in tqdm(split, desc=split_name, total=len(split)):
            seen += 1
            diff = row.get("diff") or ""
            message = row.get("message") or ""
            if not diff or not message:
                continue
            example = build_example(diff, message)
            if example is None:
                continue
            f.write(json.dumps(example) + "\n")
            kept += 1
            if LIMIT_PER_SPLIT and kept >= LIMIT_PER_SPLIT:
                break
    print(f"{split_name}: kept {kept}/{seen} ({kept / max(seen, 1):.1%}) -> {out_path}")


# --- Sanity check a few examples ---
sample_path = OUT_DIR / "train.jsonl"
with sample_path.open() as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        row = json.loads(line)
        print("MESSAGE:", row["message"])
        print("DIFF:", row["diff"][:300], "...\n")


# --- Kaggle: zip output for download ---
if ON_KAGGLE:
    shutil.make_archive("/kaggle/working/commitbench-filtered", "zip", OUT_DIR)
    print("Wrote /kaggle/working/commitbench-filtered.zip")